# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

## 1. My rule and its reason code

**Lane:** Freestyle — Diagnosis-First Content Triage.

**Two signals to check first, both computable from `fact_content_daily_performance`'s real, verified schema (no unverified join to `dim_content` needed):**

1. **Signal 1 (flag-linked -- behind the CTR-fix logic):** *Claim:* pages ranking in the top 10 should show meaningfully higher CTR than pages ranking 11th or worse, since users click top results far more often. This is the exact assumption behind FlyRank's `low_ctr_visible_page` / CTR-fix flag.
2. **Signal 2 (flag-linked -- behind quick-win logic):** *Claim:* pages sitting in 'striking distance' (position 11-30, not page 1, not deep) already carry real, non-trivial impression volume -- meaning a small ranking push could pay off. This is the exact assumption behind a 'quick win' flag.

Both are checked with grouped bucket tables and a printed n per bucket, using the CTR-weighting rule from `auditing-signals` (SUM(clicks)/SUM(impressions), never the mean of per-row CTRs) and a minimum-volume floor to avoid noise from tiny buckets.

In [1]:
# Setup — run this in Colab with HF_TOKEN stored as a Secret (never pasted in a cell; this repo is public).
%pip -q install duckdb
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

DAILY = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"
MONTH = '2026-03'  # same mid-panel month as ML-04, per the card's warning to never develop on the final month

print(con.sql(f"SELECT COUNT(*) AS n FROM {DAILY} WHERE month = '{MONTH}'").df())

         n
0  9841378


In [2]:
# Signal 1 -- CTR by position bucket (top10 vs 11+), volume-floored, weighted CTR (not mean of rates)
signal1 = con.sql(f"""
    SELECT
        CASE WHEN gsc_avg_position > 0 AND gsc_avg_position <= 10 THEN 'top_10'
             WHEN gsc_avg_position > 10 THEN '11_plus'
             ELSE 'no_position_data' END AS position_bucket,
        SUM(gsc_clicks)::DOUBLE / NULLIF(SUM(gsc_impressions), 0) AS weighted_ctr,
        SUM(gsc_impressions) AS total_impressions,
        COUNT(*) AS n
    FROM {DAILY}
    WHERE month = '{MONTH}' AND gsc_impressions >= 10  -- floor out near-zero-impression noise
    GROUP BY position_bucket
    ORDER BY position_bucket
""").df()
print(signal1)
print()
print("VERDICT: CONFIRMED.")
print("top_10 weighted CTR = 0.339% vs 11_plus weighted CTR = 0.193% -- top-10 pages convert roughly")
print("1.76x better, backed by real volume (n=1,336,703 top_10 vs n=803,878 11_plus). This confirms")
print("the assumption behind FlyRank's CTR-fix flag: ranking position genuinely predicts click-through.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    position_bucket  weighted_ctr  total_impressions        n
0           11_plus      0.001930         86288691.0   803878
1  no_position_data      0.001134           138499.0     6948
2            top_10      0.003390        188876660.0  1336703

VERDICT: CONFIRMED.
top_10 weighted CTR = 0.339% vs 11_plus weighted CTR = 0.193% -- top-10 pages convert roughly
1.76x better, backed by real volume (n=1,336,703 top_10 vs n=803,878 11_plus). This confirms
the assumption behind FlyRank's CTR-fix flag: ranking position genuinely predicts click-through.


In [3]:
# Signal 2 -- impression volume by position bucket, checking the 'striking distance' (11-30) claim
signal2 = con.sql(f"""
    SELECT
        CASE WHEN gsc_avg_position > 0 AND gsc_avg_position <= 10 THEN 'top_10'
             WHEN gsc_avg_position > 10 AND gsc_avg_position <= 30 THEN 'striking_distance_11_30'
             WHEN gsc_avg_position > 30 THEN 'deep_30_plus'
             ELSE 'no_position_data' END AS position_bucket,
        MEDIAN(gsc_impressions) AS median_impressions,
        SUM(gsc_impressions) AS total_impressions,
        COUNT(*) AS n
    FROM {DAILY}
    WHERE month = '{MONTH}'
    GROUP BY position_bucket
    ORDER BY position_bucket
""").df()
print(signal2)
print()
print("VERDICT: CONFIRMED.")
print("striking_distance_11_30 shows a median of 20 impressions/day and 56.1M total impressions across")
print("n=822,477 rows -- real, non-trivial volume, not a near-zero edge case. Pages just outside page 1")
print("are genuinely carrying meaningful search demand already, supporting the quick-win premise.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

           position_bucket  median_impressions  total_impressions        n
0             deep_30_plus                 6.0         32646608.0   605100
1         no_position_data                 0.0           468746.0  6393506
2  striking_distance_11_30                20.0         56152274.0   822477
3                   top_10                23.0        191389961.0  2020295

VERDICT: CONFIRMED.
striking_distance_11_30 shows a median of 20 impressions/day and 56.1M total impressions across
n=822,477 rows -- real, non-trivial volume, not a near-zero edge case. Pages just outside page 1
are genuinely carrying meaningful search demand already, supporting the quick-win premise.


**The rule, encoded simply and transparently (readable on purpose, no fitted weights), built directly from the two signals above:**

```
in_striking_distance = (avg_position > 10) AND (avg_position <= 30)
has_real_volume      = (impressions >= 100)
quick_win_flag       = in_striking_distance AND has_real_volume
score                = impressions * quick_win_flag   # zero for non-candidates, on purpose
```

**Reason code (single, fixed):** `striking_distance_with_volume`
**Action label (single, fixed):** `review_for_ctr_and_position_lift`

This rule is a direct encoding of Signal 2's claim (striking-distance pages with real volume are worth a look), and it is *justified* by Signal 1 (if CTR really does jump at the top-10 boundary, then nudging a striking-distance page onto page 1 has a real, evidenced payoff -- not just a hopeful guess).

## 2. Build the ranked queue (writes the CSV)

First aggregate the daily fact table up to one row per content item for the month (an editor reviews PAGES, not page-days), then apply the rule.

In [4]:
content_month = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        AVG(gsc_avg_position) AS avg_position  -- simple mean across the month's days; a known simplification,
                                                 -- not impression-weighted -- named explicitly as a limitation below
    FROM {DAILY}
    WHERE month = '{MONTH}'
    GROUP BY client_hash_id, content_hash_id
""").df()

content_month['in_striking_distance'] = (content_month['avg_position'] > 10) & (content_month['avg_position'] <= 30)
content_month['has_real_volume'] = content_month['impressions'] >= 100
content_month['quick_win_flag'] = content_month['in_striking_distance'] & content_month['has_real_volume']
content_month['score'] = content_month['impressions'] * content_month['quick_win_flag']
content_month['reason_code'] = content_month['quick_win_flag'].map(
    {True: 'striking_distance_with_volume', False: 'not_flagged'})
content_month['action_label'] = content_month['quick_win_flag'].map(
    {True: 'review_for_ctr_and_position_lift', False: 'no_action'})

queue = content_month.sort_values('score', ascending=False).reset_index(drop=True)
print(f"Total content items this month: {len(queue):,}")
print(f"Flagged as quick-win candidates: {queue['quick_win_flag'].sum():,}")
print(queue.head(10))

import os
os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print("Written to work/outputs/baseline_action_score.csv (stays out of git by design -- CI leak-guard).")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total content items this month: 331,437
Flagged as quick-win candidates: 32,954
            client_hash_id           content_hash_id  impressions  clicks  \
0  client_23a62021009f63c4  content_e8a52cf3d5988c07     244931.0   669.0   
1  client_20259bd6705d81d4  content_82e35c4845e6c391     143907.0    60.0   
2  client_23a62021009f63c4  content_3df3f32f3fd58dea     140156.0   197.0   
3  client_23a62021009f63c4  content_66288edeb93b7c4f     137878.0   782.0   
4  client_23a62021009f63c4  content_df47d1b976106de4     131707.0   163.0   
5  client_23a62021009f63c4  content_5e1c049f62e33b11     120175.0   168.0   
6  client_23a62021009f63c4  content_661a7734f691bef5     110424.0    73.0   
7  client_23a62021009f63c4  content_cae701a83cad5e36      98572.0   242.0   
8  client_20259bd6705d81d4  content_9fff53e827550f9d      94673.0   475.0   
9  client_fef1a8f436438636  content_ba462518dad435fc      91391.0    46.0   

   avg_position  in_striking_distance  has_real_volume  quick_win_flag  

## 3. Top-10 review

For each of your REAL top 10 (printed above), fill in one line: the action, why it's there, and what would make it wrong. Template below -- replace every [FILL IN] once you see real rows. Do not invent plausible-sounding text; look at the actual row.

In [5]:
top10 = queue.head(10)
reviews = {
    1: {
        "why": "Largest volume in the flagged set (244,931 impressions) at position 15, just outside page 1. Its own CTR (0.27%) sits between the top_10 (0.339%) and 11_plus (0.193%) benchmarks -- a real, proportionate opportunity if pushed to page 1.",
        "wrong": "If this monthly average of 15 hides day-to-day swings (some days on page 1, some far off), a position lift may already be happening naturally -- the simple (non-weighted) average could be masking that.",
    },
    2: {
        "why": "Second-highest volume, but CTR (0.042%) is dramatically BELOW even the 11_plus benchmark (0.193%) -- a real, large gap independent of position, suggesting a title/meta problem, not just a ranking one.",
        "wrong": "If this page's intent is one where an AI Overview or featured snippet already answers the query (see the SERP-interception framework), pushing position alone won't fix low CTR.",
    },
    3: {
        "why": "Below-benchmark CTR (0.141% vs 0.193%) at a middling striking-distance position -- a moderate, real opportunity.",
        "wrong": "The gap to the benchmark is small; if the true value is noisy, this may not be meaningfully different from a typical page at this position.",
    },
    4: {
        "why": "CTR (0.567%) is already ABOVE the top_10 benchmark (0.339%) despite sitting at position 18.6 -- this page is over-performing for its position.",
        "wrong": "This is a genuine weak pick: CTR is already excellent here. A CTR-focused review would likely waste editor time -- the real constraint is pure ranking position, not the content or the CTR.",
    },
    5: {
        "why": "Below-benchmark CTR (0.124%) at a further-out striking-distance position -- a real gap worth checking.",
        "wrong": "At position ~24, this sits near the edge of the striking-distance band (11-30); if the position estimate is noisy, this could really be a 'deep' page misclassified.",
    },
    6: {
        "why": "Below-benchmark CTR (0.140%) at a relatively strong position (18.1) -- a meaningful gap given how close it already is to page 1.",
        "wrong": "If a title/meta change is already in flight (not visible in this snapshot), this flag could already be stale.",
    },
    7: {
        "why": "Very low CTR (0.066%), well below the 11_plus benchmark -- one of the clearer real gaps in the top 10.",
        "wrong": "If most impressions come from a query type where the answer is already visible without a click, low CTR here might be structurally expected, not fixable.",
    },
    8: {
        "why": "CTR (0.246%) is actually above the 11_plus benchmark despite the below-page-1 position -- a real but smaller opportunity than the clearest gaps in this list.",
        "wrong": "Ranked this high mainly by raw impression volume, not a clear CTR problem -- the score is impression-driven here, not gap-driven, which is a real limitation of this simple rule.",
    },
    9: {
        "why": "Very high CTR (0.502%), well above even the top_10 benchmark -- like row 4, this page is already outperforming expectations for its position.",
        "wrong": "Another genuine weak pick: flagged mainly by volume, not by any real CTR problem. A pure position/technical push, not a CTR-focused content review, is the more accurate action.",
    },
    10: {
        "why": "Lowest CTR in the entire top 10 (0.050%), far below the 11_plus benchmark, at the deepest position in the striking-distance band -- a strong, real CTR-gap candidate.",
        "wrong": "At position ~27, this is closest to falling out of the striking-distance band into 'deep' -- if the true position is often beyond 30, the quick-win framing may be overly optimistic.",
    },
}
for i, row in top10.iterrows():
    r = reviews[i + 1]
    print(f"#{i+1} | content_hash_id={row['content_hash_id']} | score={row['score']:.0f} | "
          f"impressions={row['impressions']:.0f} | avg_position={row['avg_position']:.1f}")
    print("   Action: review_for_ctr_and_position_lift")
    print(f"   Why it's here: {r['why']}")
    print(f"   What would make it wrong: {r['wrong']}")
    print()


#1 | content_hash_id=content_e8a52cf3d5988c07 | score=244931 | impressions=244931 | avg_position=15.0
   Action: review_for_ctr_and_position_lift
   Why it's here: Largest volume in the flagged set (244,931 impressions) at position 15, just outside page 1. Its own CTR (0.27%) sits between the top_10 (0.339%) and 11_plus (0.193%) benchmarks -- a real, proportionate opportunity if pushed to page 1.
   What would make it wrong: If this monthly average of 15 hides day-to-day swings (some days on page 1, some far off), a position lift may already be happening naturally -- the simple (non-weighted) average could be masking that.

#2 | content_hash_id=content_82e35c4845e6c391 | score=143907 | impressions=143907 | avg_position=22.6
   Action: review_for_ctr_and_position_lift
   Why it's here: Second-highest volume, but CTR (0.042%) is dramatically BELOW even the 11_plus benchmark (0.193%) -- a real, large gap independent of position, suggesting a title/meta problem, not just a ranking one.
   

## 4. Weak picks + leakage check

**Leakage check, stated explicitly:** every input to this rule (`gsc_impressions`, `gsc_avg_position`, both summed/averaged only within the SAME month being scored) is a same-window, directly observed signal -- none of it is a future window, and none of it is derived from `trend_direction`/`trend_pct` or any FlyRank product flag (`health_score`, `priority_score`, `action_type`), which are never present in this table at all (confirmed via the real `DESCRIBE` in ML-04).

In [6]:
# Weak-pick check: rows #4 and #9 are the clearest weak picks.
print("Weak pick: #9 (content_9fff53e827550f9d), CTR = 0.502%, well ABOVE the top_10 benchmark (0.339%).")
print("Why it might be wrong: this page is already converting better than a typical top-10 page despite")
print("sitting at position 22.5 -- its CTR is not the problem. It was flagged mainly because of its raw")
print("impression volume (94,673), not because of any real CTR gap. Row #4 shows the same pattern")
print("(CTR 0.567%, also above top_10 benchmark) and is a second, equally valid weak pick.")
print()
print("What this reveals about the rule: the score (impressions * quick_win_flag) rewards raw volume")
print("alone once a page clears the striking-distance + volume gate -- it does not distinguish a real")
print("CTR gap (rows #2, #7, #10) from a page that is already performing well (rows #4, #9). A stronger")
print("Week-5 rule should multiply by an actual CTR gap (ctr_vs_expected_gap), not just impressions.")


Weak pick: #9 (content_9fff53e827550f9d), CTR = 0.502%, well ABOVE the top_10 benchmark (0.339%).
Why it might be wrong: this page is already converting better than a typical top-10 page despite
sitting at position 22.5 -- its CTR is not the problem. It was flagged mainly because of its raw
impression volume (94,673), not because of any real CTR gap. Row #4 shows the same pattern
(CTR 0.567%, also above top_10 benchmark) and is a second, equally valid weak pick.

What this reveals about the rule: the score (impressions * quick_win_flag) rewards raw volume
alone once a page clears the striking-distance + volume gate -- it does not distinguish a real
CTR gap (rows #2, #7, #10) from a page that is already performing well (rows #4, #9). A stronger
Week-5 rule should multiply by an actual CTR gap (ctr_vs_expected_gap), not just impressions.


## Named limitation

**`avg_position` is a simple mean across the month's days, not impression-weighted.** A page with one unusually bad day (e.g. a temporary drop to position 90) pulls its monthly average down even if most days were fine -- a weighted average (weighted by that day's impressions) would be more correct but adds complexity this baseline deliberately skips. Worth revisiting in the Week-5 model if this rule's top picks look distorted by single-day outliers.

## Self-check

Before you submit, confirm each line honestly:

- [x] Both signal verdicts filled in with real numbers, at least one flag-linked
- [x] The rule's score/reason_code/action_label are visible in the real queue output
- [x] work/outputs/baseline_action_score.csv actually written (check the print confirmation)
- [x] All ten top-10 lines filled in with real observations, not placeholder text
- [x] No future-window or label-derived inputs -- confirmed above
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.